# Ridge 최종 모델
ZIP의 원본 Ridge 실험을 저장소 경로에 맞게 정리한 공개용 사본입니다.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error


In [ ]:
df = pd.read_csv('../data/processed/df_final1.csv')
target = 'Book-Rating'
text_cols = ['Book-Title', 'Book-Author', 'Publisher', 'Location_country']
num_cols = ['Age', 'Year-Of-Publication']
X_train, X_test, y_train, y_test = train_test_split(df[text_cols + num_cols], df[target], test_size=0.2, random_state=42)


In [ ]:
def text_pipe(n):
    return Pipeline([('tfidf', TfidfVectorizer(min_df=3, max_features=30000, ngram_range=(1,2))), ('svd', TruncatedSVD(n_components=n, random_state=42))])
prep = ColumnTransformer([
    ('title', text_pipe(128), 'Book-Title'),
    ('author', text_pipe(64), 'Book-Author'),
    ('publisher', text_pipe(32), 'Publisher'),
    ('country', text_pipe(16), 'Location_country'),
    ('num', StandardScaler(), num_cols),
])
pipe = Pipeline([('prep', prep), ('model', Ridge())])
grid = GridSearchCV(pipe, {'model__alpha':[1, 3, 10, 30]}, scoring='neg_root_mean_squared_error', cv=KFold(3, shuffle=True, random_state=42), n_jobs=-1)
grid.fit(X_train, y_train)
pred = grid.predict(X_test)
print('best:', grid.best_params_)
print('RMSE:', root_mean_squared_error(y_test, pred))
print('MAE:', mean_absolute_error(y_test, pred))
print('R2:', r2_score(y_test, pred))
